<a href="https://colab.research.google.com/github/Srihitha-M/NASSCOM/blob/main/WEEK4_U10_Feature_Engineering_Part1_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SNAPSHOT = pd.Timestamp('2024-06-30')

def build_datasets(csv_path='ecommerce_customers.csv',
                   xlsx_path='transactions.xlsx', seed=42, verbose=False):
    rng = np.random.default_rng(seed)
    N = 2500
    start = pd.Timestamp('2021-01-01')
    end = pd.Timestamp('2024-06-30')
    horizon = (end - start).days

    cust = np.array([f'CUST{i+1:05d}' for i in range(N)])
    signup_off = rng.integers(0, horizon - 60, N)
    signup = start + pd.to_timedelta(signup_off, unit='D')

    lam = rng.gamma(2.0, 1.6, N)
    num_orders = rng.poisson(lam)

    counts = num_orders
    tot = int(counts.sum())
    cust_rep = np.repeat(cust, counts)
    signup_rep = np.repeat(signup_off, counts)
    span = np.maximum(horizon - signup_off, 1)
    span_rep = np.repeat(span, counts)
    off = (rng.random(tot) * span_rep).astype(int)
    tx_off = signup_rep + off
    tx_date = start + pd.to_timedelta(tx_off, unit='D')
    amount = rng.lognormal(3.2, 0.8, tot).round(2)
    category = rng.choice(['Electronics', 'Fashion', 'Grocery', 'Home', 'Books'],
                          tot, p=[.20, .30, .25, .15, .10])
    tx = pd.DataFrame({'customer_id': cust_rep, 'order_date': tx_date,
                       'amount': amount, 'category': category}).sort_values(
        ['customer_id', 'order_date']).reset_index(drop=True)

    agg = tx.groupby('customer_id').agg(
        total_spend=('amount', 'sum'),
        first_order=('order_date', 'min'),
        last_order=('order_date', 'max'),
    ).reset_index()

    df = pd.DataFrame({'customer_id': cust, 'signup_date': signup,
                       'num_orders': num_orders})
    df = df.merge(agg, on='customer_id', how='left')
    df['total_spend'] = df['total_spend'].fillna(0).round(2)

    df['age'] = np.clip(rng.normal(38, 12, N), 18, 82).round().astype(int)
    df['gender'] = rng.choice(['M', 'F', 'Other'], N, p=[.48, .48, .04])

    majors = ['Mumbai', 'Delhi', 'Bengaluru', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata']
    rare = ['Jaipur', 'Surat', 'Indore', 'Bhopal', 'Patna', 'Nagpur',
            'Kochi', 'Coimbatore', 'Visakhapatnam', 'Lucknow']
    pool = majors + rare
    w = np.array([.17, .15, .14, .12, .10, .08, .06] + [.013] * 10)
    w = w / w.sum()
    df['city'] = rng.choice(pool, N, p=w)

    df['plan'] = rng.choice(['Basic', 'Standard', 'Premium'], N, p=[.50, .35, .15])
    df['device'] = rng.choice(['Mobile', 'Desktop', 'Tablet'], N, p=[.60, .32, .08])
    df['payment_method'] = rng.choice(['Card', 'UPI', 'Wallet', 'NetBanking'],
                                      N, p=[.40, .35, .15, .10])
    df['support_tickets'] = rng.poisson(0.6, N)
    df['email_opt_in'] = rng.choice([0, 1], N, p=[.35, .65])

    last = pd.to_datetime(df['last_order'])
    days_since = (end - last).dt.days
    days_since_filled = days_since.fillna(horizon).to_numpy()
    z = (-2.75
         + 0.0019 * days_since_filled
         + 0.30 * df['support_tickets'].to_numpy()
         - 0.05 * df['num_orders'].to_numpy()
         + 0.70 * (df['num_orders'].to_numpy() == 0))
    p = 1 / (1 + np.exp(-z))
    df['is_churned'] = (rng.random(N) < p).astype(int)

    df = df.rename(columns={'first_order': 'first_order_date',
                            'last_order': 'last_order_date'})
    for c in ['signup_date', 'first_order_date', 'last_order_date']:
        df[c] = pd.to_datetime(df[c]).dt.date.astype('string')

    df = df[['customer_id', 'signup_date', 'first_order_date', 'last_order_date',
             'age', 'gender', 'city', 'plan', 'device', 'payment_method',
             'num_orders', 'total_spend', 'support_tickets', 'email_opt_in',
             'is_churned']]

    def punch(col, frac):
        idx = rng.choice(N, int(frac * N), replace=False)
        df.loc[idx, col] = np.nan
    punch('age', 0.07)
    punch('gender', 0.04)
    punch('city', 0.02)

    df.to_csv(csv_path, index=False)
    tx_out = tx.copy()
    tx_out['order_date'] = pd.to_datetime(tx_out['order_date']).dt.date.astype('string')
    tx_out.to_excel(xlsx_path, index=False)
    return df, tx_out

if not (os.path.exists('ecommerce_customers.csv') and os.path.exists('transactions.xlsx')):
    build_datasets()
    print('Generated dataset files.')
else:
    print('Found the provided dataset files.')

cust = pd.read_csv('ecommerce_customers.csv',
                   parse_dates=['signup_date', 'first_order_date', 'last_order_date'])
tx = pd.read_excel('transactions.xlsx', parse_dates=['order_date'])
print('Customers shape:', cust.shape, '| Transactions shape:', tx.shape)

rfm = tx.groupby('customer_id').agg(
    frequency=('amount', 'size'),
    monetary=('amount', 'sum'),
    avg_order_value=('amount', 'mean'),
    last_order=('order_date', 'max'),
    n_categories=('category', 'nunique'),
).reset_index()

rfm['recency_days'] = (SNAPSHOT - rfm['last_order']).dt.days

df = cust.merge(rfm.drop(columns='last_order'), on='customer_id', how='left')
df['frequency'] = df['frequency'].fillna(0)
df['monetary'] = df['monetary'].fillna(0)
df['n_categories'] = df['n_categories'].fillna(0)
df['recency_days'] = df['recency_days'].fillna(9999)

span_df = tx.groupby('customer_id')['order_date'].agg(
    first_tx=('min'),
    last_tx=('max')
).reset_index()
span_df['active_span_days'] = (span_df['last_tx'] - span_df['first_tx']).dt.days

df = df.merge(span_df[['customer_id', 'active_span_days']], on='customer_id', how='left')
df['active_span_days'] = df['active_span_days'].fillna(0)

denom = (df['frequency'] - 1).replace(0, np.nan)
df['avg_days_between_orders'] = (df['active_span_days'] / denom).fillna(0)

df['monetary_log'] = np.log1p(df['monetary'])

df['avg_order_value'] = df['avg_order_value'].fillna(0)
skew_aov_before = df['avg_order_value'].skew()
df['aov_log'] = np.log1p(df['avg_order_value'])
skew_aov_after = df['aov_log'].skew()
print(f"AOV Skew Before: {skew_aov_before:.2f} | After: {skew_aov_after:.2f}")

df['age_group'] = pd.cut(df['age'], bins=[0, 25, 40, 60, 200],
                         labels=['<25', '25-39', '40-59', '60+'])

df['recency_bucket'] = pd.cut(df['recency_days'], bins=[-1, 30, 90, 100000],
                              labels=['active', 'lapsing', 'dormant'])
print("Recency Bucket Counts:\n", df['recency_bucket'].value_counts())

df['tickets_per_order'] = df['support_tickets'] / df['frequency'].replace(0, np.nan)
df['tickets_per_order'] = df['tickets_per_order'].fillna(0)

df['spend_per_category'] = df['monetary'] / df['n_categories'].replace(0, np.nan)
df['spend_per_category'] = df['spend_per_category'].fillna(0)

tenure_months = (SNAPSHOT - df['signup_date']).dt.days / 30
df['orders_per_month'] = df['frequency'] / tenure_months.replace(0, np.nan)
df['orders_per_month'] = df['orders_per_month'].replace([np.inf, -np.inf], np.nan).fillna(0)

df['tenure_days'] = (SNAPSHOT - df['signup_date']).dt.days
df['signup_month'] = df['signup_date'].dt.month
df['signup_dow'] = df['signup_date'].dt.dayofweek
df['signup_is_weekend'] = (df['signup_dow'] >= 5).astype(int)

df['month_sin'] = np.sin(2 * np.pi * df['signup_month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['signup_month'] / 12)

df['days_since_first_order'] = (SNAPSHOT - pd.to_datetime(df['first_order_date'])).dt.days
df['days_since_first_order'] = df['days_since_first_order'].fillna(0)

df = pd.get_dummies(df, columns=['plan', 'device', 'payment_method'], drop_first=False, dtype=int)

freq = df['city'].value_counts(normalize=True)
df['city_freq'] = df['city'].map(freq).fillna(0)

df['gender'] = df['gender'].fillna('Unknown')
df = pd.get_dummies(df, columns=['gender'], dtype=int)

top_cities = df['city'].value_counts().head(7).index
df['city_grouped'] = df['city'].where(df['city'].isin(top_cities), 'Other')

df = pd.get_dummies(df, columns=['city_grouped'], dtype=int)

df['is_dormant'] = ((df['frequency'] == 0) | (df['recency_days'] > 180)).astype(int)

df['R_score'] = pd.qcut(-df['recency_days'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
df['F_score'] = pd.qcut(df['frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
df['M_score'] = pd.qcut(df['monetary'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
df['rfm_score'] = df['R_score'] + df['F_score'] + df['M_score']

q80 = df['monetary'].quantile(0.8)
df['high_value'] = (df['monetary'] >= q80).astype(int)

drop_cols = ['customer_id', 'signup_date', 'first_order_date', 'last_order_date',
             'city', 'age_group', 'recency_bucket', 'is_churned']

X = df.drop(columns=[c for c in drop_cols if c in df.columns])
X = X.select_dtypes(include='number')
y = df['is_churned']

X = X.fillna(X.median())

churn_corr = X.assign(churn=y).corr()['churn'].drop('churn').sort_values(ascending=False)

print(f"FINAL FEATURE MATRIX SHAPE: {X.shape}")
print(f"Missing Values in X: {X.isna().sum().sum()}")
print("Top 10 Feature Correlations with Churn:\n", churn_corr.head(10))

Generated dataset files.
Customers shape: (2500, 15) | Transactions shape: (8152, 4)
AOV Skew Before: 2.72 | After: -1.42
Recency Bucket Counts:
 recency_bucket
dormant    1507
lapsing     531
active      462
Name: count, dtype: int64
FINAL FEATURE MATRIX SHAPE: (2500, 53)
Missing Values in X: 0
Top 10 Feature Correlations with Churn:
 recency_days              0.515644
is_dormant                0.310526
support_tickets           0.068880
tenure_days               0.038022
city_grouped_Hyderabad    0.031792
signup_dow                0.024051
city_grouped_Pune         0.020868
payment_method_Card       0.020432
gender_Other              0.018652
plan_Standard             0.015577
Name: churn, dtype: float64
